# Deep Learning: Methodological Depth & Comparative Analysis

This notebook is part of the Deep Learning course at Istinye University, focusing on the comparative analysis of MLP models, activation functions, regularization techniques, and optimization methods.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch Version: {torch.__version__}")

## 1. Data Preparation

Perform data normalization and batching.

In [ ]:
# Transformation pipeline
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Dataset loading (Example: MNIST)
train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

## 2. Model Definition (MLP)

We use different activation functions and regularization techniques to compare results.

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, activation='relu', use_dropout=False, use_bn=False):
        super(MLPModel, self).__init__()
        
        # Activation selection
        if activation == 'relu':
            self.act = nn.ReLU()
        elif activation == 'tanh':
            self.act = nn.Tanh()
        else:
            self.act = nn.Sigmoid()
            
        # Define layers
        self.fc1 = nn.Linear(28*28, 512)
        
        # Batch Normalization used to mitigate Internal Covariate Shift
        # Normalizing layer inputs stabilizes and speeds up training.
        self.bn1 = nn.BatchNorm1d(512) if use_bn else nn.Identity()
        
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256) if use_bn else nn.Identity()
        
        self.fc3 = nn.Linear(256, 10)
        
        # Dropout used to prevent overfitting
        self.dropout = nn.Dropout(0.2) if use_dropout else nn.Identity()
        
    def forward(self, x):
        x = x.view(-1, 28*28)
        
        # First layer
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.act(x)
        x = self.dropout(x)
        
        # Second layer
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.act(x)
        x = self.dropout(x)
        
        # Output layer
        # Regarding Gradient Flow, final activations are typically left to the Loss function (e.g., CrossEntropyLoss)
        x = self.fc3(x)
        return x

## 3. Training Function and Methodology

This section includes functions for training and monitoring performance.

In [ ]:
def train_model(model, train_loader, criterion, optimizer, epochs=10):
    train_losses = []
    # Backpropagation allows updating our weights via gradient descent (e.g., SGD/Adam).
    # Loss function calculates predicted error, which is distributed backward (backprop).
    
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Gradient Flow: Calculation of gradients via loss using the Autograd mechanism
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")
        
    return train_losses

## 4. Visualization (Learning Curves)

Function to visualize training loss curves.

In [ ]:
def plot_learning_curves(losses, title="Learning Curves"):
    plt.figure(figsize=(10, 6))
    plt.plot(losses, label='Training Loss', color='royalblue', linewidth=2)
    plt.title(title, fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

## 5. Experiment Execution

Run different scenarios and compare the results.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLPModel(activation='relu', use_dropout=True, use_bn=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5) # Weight Decay = L2 Regularization

losses = train_model(model, train_loader, criterion, optimizer, epochs=5)
plot_learning_curves(losses, "ReLU with Dropout & Batch Norm")